### RAG Pipelines - Data Ingestion to Vector DB Pipeline


In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Parth\AppData\Local\Temp\ipykernel_14960\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all Pdf files in a directory"""
    all_documents= []
    pdf_dir = Path(pdf_directory)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)}")

    for pdf_file in pdf_files:
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(all_documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTOtal documents loaded: {len(documents)}")

    return all_documents

    #Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

            


Found 1
 Loaded 1 pages

TOtal documents loaded: 1


In [6]:
all_pdf_documents


[Document(metadata={'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-06T20:30:46+00:00', 'source': '..\\data\\pdf\\prashasth4.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'prashasth4.pdf', 'file_type': 'pdf'}, page_content='Prashasth Singh \uf0e0 prashasth068@gmail.com \uf08e\nF ull-stack Developer \uf095 +91-7268869676\n\uf0ac Portfolio \uf08e | \uf08c LinkedIn \uf08e | \uf09b GitHub \uf08e \uf041 Lucknow, UP\nEducation\n• Institute of Engineering and T echnology , Lucknow Lucknow, U.P., India\nB. Tech. - Computer Science and Engineering | CGPA: 8.1/10 Nov 2023 - Jun 2027\n• Bhavan’s K D K Vidhya Mandir Renukoot Sonebhadra, U.P., India\nClass XII - Physics, Chemistry, Math, Computer Science | 90.8% Apr 2021 - May 2022\nAchievements\n• LeetCode \uf08e: Knight | Highest Rating: 2039 – Global Rank 245 in BiWeekly Contest 152 (Top 0.8% globally)\n• Codeforces \uf08e: Pupil | Highest Rating: 1282 – Global Rank 2921 in Ro

In [7]:
#Text splitting get into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}..")
        print(f"MetaData: {split_docs[0].metadata}")

    return split_docs

    


In [8]:
chunks = split_documents(all_pdf_documents)

split 1 documents into 4 chunks

Example chunk:
Content: Prashasth Singh  prashasth068@gmail.com 
F ull-stack Developer  +91-7268869676
 Portfolio  |  LinkedIn  |  GitHub   Lucknow, UP
Education
• Institute of Engineering and T echnology , Lucknow..
MetaData: {'producer': 'xdvipdfmx (20240305)', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-06T20:30:46+00:00', 'source': '..\\data\\pdf\\prashasth4.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'prashasth4.pdf', 'file_type': 'pdf'}


### Embedding and Vector store

In [9]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ Load the SentenceTransformer model"""
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loaded model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str])-> np.ndarray:
        """
        Generating embeddings for a list of texts

        Args:

            texts: List of text strings to embed

        Returns:

            numpy array of embeddings with shape (len(texts), embedding_dim)
        
        """
        if not self.model:
            raise ValueError("Model Not Loaded")

        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generating embeddings with shape {embeddings.shape}")

        return embeddings


    ## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2181.67it/s]


Model Loaded successfully. Embedding dimension: 384


C:\Users\Parth\AppData\Local\Temp\ipykernel_14960\3164849252.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore


In [11]:
class VectorStore:
    """Manages document into chromadb vector store"""

    def __init__(self, collection_name:str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store
        
        Arges:
            collection_name: Name of the chromadb collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize chromadb client and collection"""

        try:
            #create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"VectorStore initialize collection {self.collection_name}")
            print(f"Exisitng document in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store:  {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        
        """
        Add documents and their embeddings to the vector store

        Args:

            documents: List of LangChain documents
            embeddings: Correspondings for the documents
        
        """
        if len(documents)!=len(embeddings):
            raise ValueError("No. of documents must match no. of embeddings")

        print(f"Adding {len(documents)} document to vectorStore...")

        #Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

            #add to collection
            try:

                self.collection.add(
                    ids=ids,
                    embeddings = embeddings_list,
                    metadatas = metadatas,
                    documents = documents_text
                )

                print(f"Successfully added {len}")



_IncompleteInputError: incomplete input (3164561501.py, line 89)